In [ ]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import glob
from Bio import SeqIO, Seq
import pandas as pd
import collections
import warnings
warnings.filterwarnings('ignore')
import shutil
from preprocess import *


# Data Preprocessing Demo for inDecay Input

This notebook demonstrates the step-by-step process of preprocessing raw sequencing data to generate model-ready inputs.



Create necessary directories for intermediate data outputs of decodr and synthego. Existing directories will be cleared and recreated for a clean preprocessing run.



In [ ]:
def create_folder(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    else:
        shutil.rmtree(folder_path)
        os.makedirs(folder_path)

create_folder('synthego/')
create_folder('decodr/')
pj=os.path.join
os.getcwd()

Set your raw data directory 

In [ ]:
rawfile_dir='raw_all'

Input folder name, guideRNA seq and sample type, it will process all the files in the folder to decodr and synthego input format;
<br>
here we use mKat2a as example
- `folder`: Raw data input folder.
- `Guide`: sgRNA guide sequence.
- `stype`: Sample type - use 'Clonal' or 'Bulk'.

In [ ]:
folder='mKat2a_20220608' ##input the raw data folder name
Guide= 'CTGCCTTAACTACTGGAAGC' ## input your guide here
requirements=[]
stype='Clonal' ##input your sample type: clonal or bulk
abi, definition_ice, definition_dec = find_ab1_and_control(folder, stype, Guide, requirements, 
                                                save_name=folder)


find sgRNA in forward strand with PAM


In [4]:
folder='mKat2a_20220621' ##input the raw data folder name
Guide= 'CTGCCTTAACTACTGGAAGC' ## input your guide here
requirements=[]
stype='Clonal' ##input your sample type: clonal or bulk
abi, definition_ice, definition_dec = find_ab1_and_control(folder, stype, Guide, requirements, 
                                                save_name=folder)


find sgRNA in forward strand with PAM


In [5]:
folder='mKat2a_20220628' ##input the raw data folder name
Guide= 'CTGCCTTAACTACTGGAAGC' ## input your guide here
requirements=[]
stype='Clonal' ##input your sample type: clonal or bulk
abi, definition_ice, definition_dec = find_ab1_and_control(folder, stype, Guide, requirements, 
                                                save_name=folder)


find sgRNA in forward strand with PAM


zip files for batch-input in synthego

In [ ]:


# --- Configuration ---
synthego = 'synthego/raw_syn'
output_dir = synthego+'_zip'
os.makedirs(output_dir, exist_ok=True)

file_list = glob.glob(os.path.join(synthego, "*.xlsx"))
max_thre = 450  # Max number of rows per batch

# --- Trackers ---
excl_list = []
f_list = []
all_exc_list = []
each_shape = 0
num = 0

# --- Start Processing Files ---
for file in file_list:
    # Keep track of previous state before updating
    excl_archive = excl_list.copy()
    f_archive = f_list.copy()
    pd_archive = pd.concat(excl_list, ignore_index=True) if excl_list else pd.DataFrame()

    # Read current file
    excl_list.append(pd.read_excel(file))
    excl_merged = pd.concat(excl_list, ignore_index=True)

    # Track folder name (filename without extension)
    folder_name = os.path.basename(file).replace('.xlsx', '')
    f_list.append(folder_name)
    all_exc_list.append(pd.read_excel(file))

    # --- Batch Condition: If merged DataFrame exceeds threshold ---
    if excl_merged.shape[0] > max_thre and not pd_archive.empty:
        # Save previous batch (excluding current file)
        prev_df = pd.concat(excl_archive, ignore_index=True)
        newdir = os.path.join(output_dir, f'round_{num}')
        os.makedirs(newdir, exist_ok=True)

        # Save Excel file
        prev_df.to_excel(os.path.join(output_dir, f'round{num}.xlsx'), index=False)

        # Copy .ab1 files
        for folder in f_archive:
            src_folder = os.path.join(synthego, folder)
            if not os.path.exists(src_folder):
                raise FileNotFoundError(f"Directory not found: {src_folder}")

            for file_name in os.listdir(src_folder):
                if file_name.endswith(".ab1"):
                    shutil.copy2(os.path.join(src_folder, file_name), newdir)

        # Zip and clean up
        shutil.make_archive(newdir, 'zip', newdir)
        shutil.rmtree(newdir)

        # Update trackers
        excl_list = [pd.read_excel(file)]
        f_list = [folder_name]
        each_shape += prev_df.shape[0]
        num += 1

# --- Final Batch: Handle any remaining files ---
if excl_list:
    final_df = pd.concat(excl_list, ignore_index=True)
    newdir = os.path.join(output_dir, f'round_{num}')
    os.makedirs(newdir, exist_ok=True)

    # Save final Excel
    final_df.to_excel(os.path.join(output_dir, f'round{num}.xlsx'), index=False)

    # Copy final .ab1 files
    for folder in f_list:
        src_folder = os.path.join(synthego, folder)
        if not os.path.exists(src_folder):
            raise FileNotFoundError(f"Directory not found: {src_folder}")

        for file_name in os.listdir(src_folder):
            if file_name.endswith(".ab1"):
                shutil.copy2(os.path.join(src_folder, file_name), newdir)

    # Zip and clean up
    shutil.make_archive(newdir, 'zip', newdir)
    shutil.rmtree(newdir)

    each_shape += final_df.shape[0]

# --- Validation Step (Optional or Replace with Logging) ---
excel_all = pd.concat(all_exc_list, ignore_index=True)

assert excel_all.shape[0] == each_shape, [
    "Excel row count mismatch",
    excel_all.shape[0],
    each_shape,
]

copied_files = glob.glob(os.path.join(output_dir, "*.ab1"))
original_files = glob.glob(os.path.join(synthego, "*.ab1"))

assert len(copied_files) == len(original_files), [
    "File count mismatch",
    len(copied_files),
    len(original_files),
]

print("✅ All batches created and validated successfully. Plz input the zip files to ice synthego and download the results")

✅ All batches created and validated successfully. Plz input the zip files to ice synthego and download the results


zip files for batch-input in decodr


In [ ]:


# --- Configuration ---
decodr = 'decodr/raw_dec'
output_dir = decodr+'_zip'
os.makedirs(output_dir, exist_ok=True)

file_list = glob.glob(os.path.join(decodr, "*.xlsx"))
max_thre = 140  # Max number of rows per batch

# --- Trackers ---
excl_list = []
f_list = []
all_exc_list = []
each_shape = 0
num = 0

# --- Start Processing Files ---
for file in file_list:
    # Keep track of previous state before updating
    excl_archive = excl_list.copy()
    f_archive = f_list.copy()
    pd_archive = pd.concat(excl_list, ignore_index=True) if excl_list else pd.DataFrame()

    # Read current file
    excl_list.append(pd.read_excel(file))
    excl_merged = pd.concat(excl_list, ignore_index=True)

    # Track folder name (filename without extension)
    
    folder_name = os.path.basename(file).replace('.xlsx', '')
    f_list.append(folder_name)
    all_exc_list.append(pd.read_excel(file))

    # --- Batch Condition: If merged DataFrame exceeds threshold ---
    if excl_merged.shape[0] > max_thre and not pd_archive.empty:
        # Save previous batch (excluding current file)
        prev_df = pd.concat(excl_archive, ignore_index=True)
        newdir = os.path.join(output_dir, f'round_{num}')
        os.makedirs(newdir, exist_ok=True)

        # Save Excel file
        prev_df['Donor Template (Optional)']= ['None']*prev_df.shape[0]
        prev_df.to_excel(os.path.join(output_dir, f'round{num}.xlsx'), index=False)

        # Copy .ab1 files
        for folder in f_archive:
            src_folder = os.path.join(decodr, folder)
            if not os.path.exists(src_folder):
                raise FileNotFoundError(f"Directory not found: {src_folder}")

            for file_name in os.listdir(src_folder):
                if file_name.endswith(".ab1"):
                    shutil.copy2(os.path.join(src_folder, file_name), newdir)

        # Zip and clean up
        shutil.make_archive(newdir, 'zip', newdir)
        shutil.rmtree(newdir)

        # Update trackers
        excl_list = [pd.read_excel(file)]
        f_list = [folder_name]
        each_shape += prev_df.shape[0]
        num += 1

# --- Final Batch: Handle any remaining files ---
if excl_list:
    final_df = pd.concat(excl_list, ignore_index=True)
    newdir = os.path.join(output_dir, f'round_{num}')
    os.makedirs(newdir, exist_ok=True)

    # Save final Excel
    final_df['Donor Template (Optional)']= ['None']*final_df.shape[0]
    final_df.to_excel(os.path.join(output_dir, f'round{num}.xlsx'), index=False)

    # Copy final .ab1 files
    for folder in f_list:
        src_folder = os.path.join(decodr, folder)
        if not os.path.exists(src_folder):
            raise FileNotFoundError(f"Directory not found: {src_folder}")

        for file_name in os.listdir(src_folder):
            if file_name.endswith(".ab1"):
                shutil.copy2(os.path.join(src_folder, file_name), newdir)

    # Zip and clean up
    shutil.make_archive(newdir, 'zip', newdir)
    shutil.rmtree(newdir)

    each_shape += final_df.shape[0]

# --- Validation Step (Optional or Replace with Logging) ---
excel_all = pd.concat(all_exc_list, ignore_index=True)

assert excel_all.shape[0] == each_shape, [
    "Excel row count mismatch",
    excel_all.shape[0],
    each_shape,
]

copied_files = glob.glob(os.path.join(output_dir, "*.ab1"))
original_files = glob.glob(os.path.join(decodr, "*.ab1"))

assert len(copied_files) == len(original_files), [
    "File count mismatch",
    len(copied_files),
    len(original_files),
]

print("✅ All batches created and validated successfully. Plz input the zip files to decodr and download the results")

✅ All batches created and validated successfully. Plz input the zip files to decodr and download the results


now, you can run `process.py` to get the final inDecay format input

In [42]:
# ! cd /home/louisayu/ssd/inDecay/zygotes/
! python process.py

[I @2025-09-02 16:47:00,909]: Extracting guide information …
[I @2025-09-02 16:47:01,726]: Total guides extracted: 994
[I @2025-09-02 16:47:01,726]: Preparing SelfTarget commands …
[I @2025-09-02 16:47:03,795]: Executing /ssd/users/louisayu/inDecay/zygotes/20250902_run_selftarget.sh (contains 118 unique st)
/ssd/users/louisayu/inDecay/zygotes
[I @2025-09-02 16:47:56,936]: Transforming decodr outputs …
[I @2025-09-02 16:48:32,140]: Transforming ice outputs …
[I @2025-09-02 16:48:32,141]: Aggregating Sanger & ICE folders …
[I @2025-09-02 16:48:33,649]: Removing rejected duplicates …
Selected mFABP4_20250417R for datemark mFABP4_20250417
Selected mH2AB1-sg1_20220412R for datemark mH2AB1-sg1_20220412
Selected mFABP4_20250418R for datemark mFABP4_20250418
Selected mH2D1-sg2_20220412R for datemark mH2D1-sg2_20220412
Selected cPRDM14_20241127F for datemark cPRDM14_20241127
Selected pAPN_20241213R for datemark pAPN_20241213
Selected pGHR_20241213F for datemark pGHR_20241213
Selected mIL21R_202

In [44]:
pd.read_csv('/home/louisayu/ssd/inDecay/data/gene_seq.csv').head()

,guide,seq,r2,count
0,cGHR,TCCAGTGACATGAGAAAGTCTCCAGTTCAGGTGAACGGCACTTGGT...,0.924007,1914.90
1,cPRDM14,AAATCTCAGGTCCCCAACACTGGCCACCGTCGCTCCGGCCGGCGGT...,0.939555,2263.80
2,cRAG2,ATGAATTTTGATGGGCAAGTTTTCTTCTTTGGCCAAAAAGGCTGGC...,0.737750,2363.10
3,gGHR,GTTTTTCTTTGTATGATTGCAGGTCCTACAGGTATGGATCTCTGGA...,0.942836,2500.00
4,gPRDM1,CGCCGAAGAAGATGTCGCAGACGAAGGAGGCGCTTTGGTCGGAGGC...,0.954071,2500.25
